In [1]:
import os

print("Jupyter is running from:")
print(os.getcwd())

print("\nDatabase is located at:")
print(os.path.abspath('../data/sales_forecast.db'))

Jupyter is running from:
C:\Users\oorko

Database is located at:
C:\Users\data\sales_forecast.db


In [3]:
import pandas as pd
import duckdb
import os

db_path = r"C:\Users\oorko\OneDrive\Documents\projects\Sales_Forecasting\data\sales_forecast.db"

conn = duckdb.connect(db_path)

print("Connected to DuckDB successfully")
print(f"Database location: {db_path}")

Connected to DuckDB successfully
Database location: C:\Users\oorko\OneDrive\Documents\projects\Sales_Forecasting\data\sales_forecast.db


In [6]:
folder = r"C:\Users\oorko\OneDrive\Documents\projects\Sales_Forecasting\data\raw"

train = pd.read_csv(folder + r"\train.csv", parse_dates=["date"])
test = pd.read_csv(folder + r"\test.csv", parse_dates=["date"])
stores = pd.read_csv(folder + r"\stores.csv")
oil = pd.read_csv(folder + r"\oil.csv", parse_dates=["date"])
holidays = pd.read_csv(folder + r"\holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(folder + r"\transactions.csv", parse_dates=["date"])

print("All CSVs loaded into pandas")

All CSVs loaded into pandas


In [7]:
# Write each dataframe as a table in DuckDB
conn.execute("DROP TABLE IF EXISTS raw_train")
conn.execute("DROP TABLE IF EXISTS raw_test")
conn.execute("DROP TABLE IF EXISTS raw_stores")
conn.execute("DROP TABLE IF EXISTS raw_oil")
conn.execute("DROP TABLE IF EXISTS raw_holidays")
conn.execute("DROP TABLE IF EXISTS raw_transactions")

conn.execute("CREATE TABLE raw_train AS SELECT * FROM train")
conn.execute("CREATE TABLE raw_test AS SELECT * FROM test")
conn.execute("CREATE TABLE raw_stores AS SELECT * FROM stores")
conn.execute("CREATE TABLE raw_oil AS SELECT * FROM oil")
conn.execute("CREATE TABLE raw_holidays AS SELECT * FROM holidays")
conn.execute("CREATE TABLE raw_transactions AS SELECT * FROM transactions")

print("All tables created in DuckDB")

All tables created in DuckDB


In [8]:
tables = conn.execute("SHOW TABLES").fetchdf()
print("Tables in DuckDB:")
print(tables)

Tables in DuckDB:
               name
0      raw_holidays
1           raw_oil
2        raw_stores
3          raw_test
4         raw_train
5  raw_transactions


In [9]:
table_names = ['raw_train', 'raw_test', 'raw_stores', 'raw_oil', 'raw_holidays', 'raw_transactions']

print("=== ROW COUNT VERIFICATION ===\n")
for table in table_names:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count:,} rows")

=== ROW COUNT VERIFICATION ===

raw_train: 3,000,888 rows
raw_test: 28,512 rows
raw_stores: 54 rows
raw_oil: 1,218 rows
raw_holidays: 350 rows
raw_transactions: 83,488 rows


In [10]:
# Check train table
print("=== TRAIN TABLE SAMPLE ===")
print(conn.execute("SELECT * FROM raw_train LIMIT 5").fetchdf())

# Check date range in SQL
print("\n=== DATE RANGE ===")
print(conn.execute("""
    SELECT 
        MIN(date) as earliest_date,
        MAX(date) as latest_date,
        COUNT(DISTINCT date) as unique_dates
    FROM raw_train
""").fetchdf())

# Check for nulls in sales column
print("\n=== NULL CHECK ===")
print(conn.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(sales) as non_null_sales,
        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) as null_sales
    FROM raw_train
""").fetchdf())

=== TRAIN TABLE SAMPLE ===
   id       date  store_nbr      family  sales  onpromotion
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0
1   1 2013-01-01          1   BABY CARE    0.0            0
2   2 2013-01-01          1      BEAUTY    0.0            0
3   3 2013-01-01          1   BEVERAGES    0.0            0
4   4 2013-01-01          1       BOOKS    0.0            0

=== DATE RANGE ===
  earliest_date latest_date  unique_dates
0    2013-01-01  2017-08-15          1684

=== NULL CHECK ===
   total_rows  non_null_sales  null_sales
0     3000888         3000888         0.0


In [11]:
print("=== OIL PRICE NULL CHECK ===")
print(conn.execute("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN dcoilwtico IS NULL THEN 1 ELSE 0 END) as null_prices,
        ROUND(100.0 * SUM(CASE WHEN dcoilwtico IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_null
    FROM raw_oil
""").fetchdf())

=== OIL PRICE NULL CHECK ===
   total_rows  null_prices  pct_null
0        1218         43.0      3.53


In [13]:
# This is a preview of the final joined table you'll create in Phase 4
preview = conn.execute("""
    SELECT 
        t.date,
        t.store_nbr,
        t.family,
        t.sales,
        t.onpromotion,
        s.city,
        s.state,
        s.type as store_type,
        s.cluster,
        o.dcoilwtico as oil_price,
        tr.transactions
    FROM raw_train t
    LEFT JOIN raw_stores s ON t.store_nbr = s.store_nbr
    LEFT JOIN raw_oil o ON t.date = o.date
    LEFT JOIN raw_transactions tr ON t.date = tr.date AND t.store_nbr = tr.store_nbr
    LIMIT 10
""").fetchdf()

print("=== PREVIEW OF FINAL JOINED TABLE ===")
print(preview)
print(f"\nColumns: {list(preview.columns)}")

=== PREVIEW OF FINAL JOINED TABLE ===
        date  store_nbr                      family    sales  onpromotion  \
0 2014-07-19         16                   MAGAZINES    3.000            0   
1 2014-07-19         16                       MEATS   61.114            0   
2 2014-07-19         16               PERSONAL CARE  279.000            0   
3 2014-07-19         16                PET SUPPLIES    0.000            0   
4 2014-07-19         16     PLAYERS AND ELECTRONICS    4.000            0   
5 2014-07-19         16                     POULTRY  137.496            0   
6 2014-07-19         16              PREPARED FOODS   56.000            0   
7 2014-07-19         16                     PRODUCE  797.368            5   
8 2014-07-19         16  SCHOOL AND OFFICE SUPPLIES    0.000            0   
9 2014-07-19         16                     SEAFOOD    6.000            0   

            city                           state store_type  cluster  \
0  Santo Domingo  Santo Domingo de los Tsa

In [15]:
conn.close()
print("Connection closed. Database saved.")

Connection closed. Database saved.


In [17]:
import os

db_path = r"C:\Users\oorko\OneDrive\Documents\projects\Sales_Forecasting\data\sales_forecast.db"

size_mb = os.path.getsize(db_path) / (1024 * 1024)

print(f"Database file: {os.path.abspath(db_path)}")
print(f"File size: {size_mb:.1f} MB")

Database file: C:\Users\oorko\OneDrive\Documents\projects\Sales_Forecasting\data\sales_forecast.db
File size: 17.5 MB
